# LENS — EfficientNet-B7 Forensic Training
**Streams CIFAKE from HuggingFace | Trains on T4 GPU | Saves to Google Drive**

Runtime → Change runtime type → T4 GPU → Save

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — change runtime!')
print('CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Go to Runtime > Change runtime type > GPU'

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────────────────
!pip install -q timm torchmetrics albumentations datasets huggingface_hub

In [ ]:
# ── Cell 3: Mount Google Drive (checkpoints saved here) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/LENS/checkpoints/efficientnet'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CKPT_DIR)

In [ ]:
# ── Cell 4: HuggingFace token ─────────────────────────────────────────────────
from huggingface_hub import login
# Paste your NEW token below (revoke old one at huggingface.co/settings/tokens)
HF_TOKEN = 'hf_YOUR_NEW_TOKEN_HERE'
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to HuggingFace')

In [ ]:
# ── Cell 5: Model definition ──────────────────────────────────────────────────
import torch, torch.nn as nn, torch.nn.functional as F, timm

FAMILY_CLASSES = ['REAL', 'GAN', 'Diffusion', 'Neural']

class PatchHead(nn.Module):
    def __init__(self, c=2560):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(c, 256, 2, 2), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 1, 1)
        )
    def forward(self, x): return self.up(x).flatten(1)

class EfficientNetForensic(nn.Module):
    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b7.ns_jft_in1k',
            pretrained=pretrained, features_only=True, out_indices=(2,3,4)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        ch = self.backbone.feature_info.channels()
        self.lat = nn.Sequential(
            nn.Conv2d(ch[0]+ch[1], ch[2], 1, bias=False),
            nn.BatchNorm2d(ch[2]), nn.GELU()
        )
        C = ch[2]
        self.binary = nn.Sequential(nn.Linear(C,512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout), nn.Linear(512,1))
        self.family = nn.Sequential(nn.Linear(C,512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout), nn.Linear(512,4))
        self.patch  = PatchHead(C)

    def forward(self, x):
        f0, f1, f2 = self.backbone(x)
        h, w = f0.shape[2:]
        fused = self.lat(torch.cat([f0, F.interpolate(f1,(h,w),mode='bilinear',align_corners=False)], 1))
        g = self.pool(f2).flatten(1)
        return {'binary': self.binary(g), 'family': self.family(g), 'patch': self.patch(fused)}

class ForensicLoss(nn.Module):
    def __init__(self, lb=1.0, lf=0.5, lp=0.3):
        super().__init__()
        self.lb, self.lf, self.lp = lb, lf, lp
        self.bce = nn.BCEWithLogitsLoss()
        self.ce  = nn.CrossEntropyLoss(label_smoothing=0.05)

    def forward(self, p, labels, families):
        l_bin   = self.bce(p['binary'].squeeze(1), labels.float())
        l_fam   = self.ce(p['family'], families)
        pt      = labels.float().unsqueeze(1).expand_as(p['patch'])
        l_patch = self.bce(p['patch'], pt)
        loss    = self.lb*l_bin + self.lf*l_fam + self.lp*l_patch
        return {'loss': loss, 'l_bin': l_bin, 'l_fam': l_fam}

print('Model classes defined OK')

In [ ]:
# ── Cell 6: Dataset — stream CIFAKE from HuggingFace ─────────────────────────
import numpy as np
import albumentations as A
import cv2
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset

FAMILY_TO_IDX = {f: i for i, f in enumerate(FAMILY_CLASSES)}
IMG_SIZE = 224

def build_aug(train=True):
    if train:
        return A.Compose([
            A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.75,1.0), p=1.0),
            A.HorizontalFlip(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.4),
            A.GaussianBlur(blur_limit=(3,7), p=0.3),
            A.GaussNoise(std_range=(0.01, 0.05), p=0.15),
            A.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5), max_pixel_value=255.0),
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5), max_pixel_value=255.0),
    ])

class CIFAKEStream(IterableDataset):
    def __init__(self, split, max_samples=None):
        self.ds = load_dataset('dragonintelligence/CIFAKE-image-dataset',
                               split=split, streaming=True, token=HF_TOKEN)
        self.aug = build_aug(split=='train')
        self.max = max_samples

    def __iter__(self):
        count = 0
        for item in self.ds.shuffle(buffer_size=1000):
            if self.max and count >= self.max: break
            try:
                img = item['image']
                if hasattr(img, 'convert'): img = img.convert('RGB')
                arr = np.array(img)
                if arr.shape[:2] != (IMG_SIZE, IMG_SIZE):
                    arr = cv2.resize(arr, (IMG_SIZE, IMG_SIZE))
                arr = self.aug(image=arr)['image'].astype(np.float32)
                t = torch.from_numpy(arr.transpose(2,0,1))
                label = int(item['label'])
                yield t, label, 1 if label == 1 else 0
                count += 1
            except: continue

print('Dataset class defined. Testing stream...')
ds_test = CIFAKEStream('train', max_samples=4)
for imgs, labels, fam in DataLoader(ds_test, batch_size=4):
    print(f'Batch: imgs={imgs.shape}, labels={labels}, fam={fam}')
    break
print('Stream OK!')

In [ ]:
# ── Cell 7: Training config ───────────────────────────────────────────────────
CFG = dict(
    epochs       = 20,
    batch_size   = 32,       # T4 has 16GB — safe for B7
    lr           = 1e-4,
    weight_decay = 1e-5,
    warmup_steps = 200,
    grad_clip    = 1.0,
    train_steps  = 3000,     # steps per epoch (CIFAKE has 100k train images)
    val_samples  = 5000,
    patience     = 5,
)
print('Config:', CFG)

In [ ]:
# ── Cell 8: Training loop ─────────────────────────────────────────────────────
import math, time
from torchmetrics.classification import BinaryAUROC
from pathlib import Path

device = torch.device('cuda')
model  = EfficientNetForensic(pretrained=True, dropout=0.3).to(device)
crit   = ForensicLoss()
opt    = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scaler = torch.cuda.amp.GradScaler()

# Cosine LR with warmup
total_steps = CFG['epochs'] * CFG['train_steps']
def lr_lambda(step):
    if step < CFG['warmup_steps']:
        return step / max(1, CFG['warmup_steps'])
    progress = (step - CFG['warmup_steps']) / max(1, total_steps - CFG['warmup_steps'])
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

# AUC tracker
auc_metric = BinaryAUROC().to(device)

best_auc   = 0.0
no_improve = 0
global_step= 0

for epoch in range(CFG['epochs']):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loader = DataLoader(
        CIFAKEStream('train', max_samples=CFG['train_steps']*CFG['batch_size']),
        batch_size=CFG['batch_size'], num_workers=2
    )
    total_loss, steps = 0.0, 0
    t0 = time.time()

    for imgs, labels, families in train_loader:
        imgs     = imgs.to(device)
        labels   = labels.to(device)
        families = families.to(device)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            out    = model(imgs)
            losses = crit(out, labels, families)

        scaler.scale(losses['loss']).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(opt)
        scaler.update()
        scheduler.step()

        total_loss += losses['loss'].item()
        steps += 1
        global_step += 1

        if steps % 100 == 0:
            print(f'  E{epoch} step {steps}/{CFG["train_steps"]} | loss={total_loss/steps:.4f} | lr={scheduler.get_last_lr()[0]:.2e}')

    # ── Validate ────────────────────────────────────────────────────────────
    model.eval()
    auc_metric.reset()
    val_loader = DataLoader(
        CIFAKEStream('test', max_samples=CFG['val_samples']),
        batch_size=CFG['batch_size']*2, num_workers=2
    )
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)
            with torch.cuda.amp.autocast():
                out = model(imgs)
            auc_metric.update(out['binary'].squeeze(1), labels)

    val_auc = auc_metric.compute().item()
    elapsed = time.time() - t0
    print(f'\nEpoch {epoch:02d} | train_loss={total_loss/steps:.4f} | val_auc={val_auc:.4f} | {elapsed:.0f}s')

    # ── Save checkpoint ─────────────────────────────────────────────────────
    ckpt = {
        'epoch': epoch, 'model': model.state_dict(),
        'optimizer': opt.state_dict(), 'val_auc': val_auc
    }
    torch.save(ckpt, f'{CKPT_DIR}/epoch_{epoch:02d}_auc{val_auc:.4f}.pth')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(ckpt, f'{CKPT_DIR}/best.pth')
        print(f'  *** New best AUC: {best_auc:.4f} — saved to {CKPT_DIR}/best.pth')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= CFG['patience']:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nTraining complete. Best AUC: {best_auc:.4f}')

In [ ]:
# ── Cell 9: Export best model to ONNX ────────────────────────────────────────
import subprocess
best_path = f'{CKPT_DIR}/best.pth'

ckpt = torch.load(best_path, map_location='cpu')
model_export = EfficientNetForensic(pretrained=False).eval()
model_export.load_state_dict(ckpt['model'])

dummy = torch.randn(1, 3, 224, 224)
onnx_path = f'{CKPT_DIR}/efficientnet_b7_forensic.onnx'

class ExportWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): o = self.m(x); return o['binary'], o['family'], o['patch']

with torch.no_grad():
    torch.onnx.export(
        ExportWrapper(model_export), dummy, onnx_path,
        opset_version=17,
        input_names=['image'],
        output_names=['binary_logit','family_logit','patch_logit'],
        dynamic_axes={'image':{0:'batch'},'binary_logit':{0:'batch'}},
        do_constant_folding=True
    )

import onnx
onnx.checker.check_model(onnx_path)
size_mb = Path(onnx_path).stat().st_size / 1e6
print(f'ONNX exported: {onnx_path}  ({size_mb:.1f} MB)')
print(f'Best val AUC: {ckpt["val_auc"]:.4f}')

In [ ]:
# ── Cell 10: Download best.pth to local machine ───────────────────────────────
from google.colab import files
print('Files in checkpoint dir:')
for f in Path(CKPT_DIR).glob('*'):
    print(f'  {f.name}  {f.stat().st_size/1e6:.1f} MB')

# Download best checkpoint
files.download(f'{CKPT_DIR}/best.pth')
files.download(onnx_path)